In [0]:
# Databricks notebook source --> Version: 03
# Auto Loader: Landing -> Bronze (micro-batch)
# - Ingesta Excel (.xlsx), Drawings (.DWG) y Models3D (.STEP)
# - Copia a Bronze y (opcional) elimina de Landing

# COMMAND ----------
# Configuracion base (constantes)
STORAGE_ACCOUNT = "datablobstorage001"
CONTAINER = "blobstorage"

# Rutas Landing (constantes)
LANDING_EXCEL = "/landingzone/Excel"
LANDING_DRAWINGS = "/landingzone/Drawings"
LANDING_MODELS = "/landingzone/Models3D"
LANDING_DELIVERY_NOTES = "/landingzone/Deliver-Notes-1001.csv"

# Rutas Bronze y configuracion UC (constantes)
BRONZE_ROOT = "/bronze"
BRONZE_LANDING_ROOT = "/bronze/_landing"
DELETE_FROM_LANDING = True
CLEAR_CHECKPOINT_BEFORE_RUN = True
UC_CATALOG = "parts"
UC_SCHEMA = "bronze"

# Metadatos de job (run_id opcional)
dbutils.widgets.text("run_id", "")

# COMMAND ----------
# Helpers
landing_excel = LANDING_EXCEL
landing_drawings = LANDING_DRAWINGS
landing_models = LANDING_MODELS
landing_delivery_notes = LANDING_DELIVERY_NOTES

bronze_root = BRONZE_ROOT
bronze_landing_root = BRONZE_LANDING_ROOT
uc_catalog = UC_CATALOG
uc_schema = UC_SCHEMA
run_id = dbutils.widgets.get("run_id")
delete_from_landing = DELETE_FROM_LANDING

# ABFSS base
abfss_base = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

landing_excel_path = f"{abfss_base}{landing_excel}"
landing_drawings_path = f"{abfss_base}{landing_drawings}"
landing_models_path = f"{abfss_base}{landing_models}"
landing_delivery_notes_path = f"{abfss_base}{landing_delivery_notes}"

# Leer Delivery-Notes.csv para obtener Date y Batch-Number
delivery_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(landing_delivery_notes_path)
)

log_lines = []
def log(msg):
    from datetime import datetime
    ts = datetime.utcnow().isoformat()
    line = f"{ts} | {msg}"
    log_lines.append(line)
    print(line)

required_cols = {"Project", "Source", "Date", "Batch-Number"}
missing_cols = required_cols - set(delivery_df.columns)
if missing_cols:
    log(f"ERROR: Delivery-Notes.csv missing columns: {sorted(missing_cols)}")
    raise ValueError(f"Delivery-Notes.csv missing columns: {sorted(missing_cols)}")

delivery_row = delivery_df.limit(1).collect()
if not delivery_row:
    log("ERROR: Delivery-Notes.csv no tiene filas.")
    raise ValueError("Delivery-Notes.csv no tiene filas. No se puede obtener Date ni Batch-Number.")

delivery_row = delivery_row[0].asDict()

run_date = str(delivery_row.get("Date")).split(" ")[0]
batch_id = str(delivery_row.get("Batch-Number"))
project = str(delivery_row.get("Project", "")).strip()
source = str(delivery_row.get("Source", "")).strip()
job_id = batch_id

bronze_base = f"{abfss_base}{bronze_landing_root}/{project}/{source}/{run_date}/{batch_id}"
bronze_excel_path = f"{bronze_base}/Excel"
bronze_drawings_path = f"{bronze_base}/Drawings"
bronze_models_path = f"{bronze_base}/Models3d"
bronze_notes_path = f"{bronze_base}/Delivery_notes"

checkpoint_base = f"{abfss_base}{bronze_root}/_checkpoints/autoloader"

# Opcional: borrar checkpoint para reprocesar todo lo existente
if CLEAR_CHECKPOINT_BEFORE_RUN:
    try:
        dbutils.fs.rm(checkpoint_base, True)
        log(f"Deleted checkpoint path: {checkpoint_base}")
    except Exception as e:
        log(f"WARNING: No se pudo borrar checkpoint {checkpoint_base}: {e}")

# Asegura carpetas Bronze aunque no haya archivos en Landing
dbutils.fs.mkdirs(bronze_base)
dbutils.fs.mkdirs(bronze_excel_path)
dbutils.fs.mkdirs(bronze_drawings_path)
dbutils.fs.mkdirs(bronze_models_path)

# COMMAND ----------
# Copy helper (micro-batch)

def copy_batch(df, target_root):
    paths = [r["path"] for r in df.select("path").collect()]
    log(f"Copy batch to {target_root}: {len(paths)} files")
    for src in paths:
        name = src.split("/")[-1]
        dst = f"{target_root}/{name}"
        dbutils.fs.cp(src, dst)
        if delete_from_landing:
            dbutils.fs.rm(src)

# COMMAND ----------
# Escribir delivery note en Bronze
log(f"Resolved Project={project}, Source={source}, Date={run_date}, Batch={batch_id}")

note_name = f"Note-date_{run_date}_batch_{batch_id}.txt"
note_dst = f"{bronze_notes_path}/{note_name}"
note_content = (
    f"Project: {project}\n"
    f"Source: {source}\n"
    f"Date: {run_date}\n"
    f"Batch-Number: {batch_id}\n"
)
dbutils.fs.mkdirs(bronze_notes_path)
dbutils.fs.put(note_dst, note_content, overwrite=True)
log(f"Wrote delivery note: {note_dst}")

# Mover Delivery-Notes-<xxxx>.csv a Bronze
delivery_csv_name = landing_delivery_notes_path.split("/")[-1]
delivery_csv_dst = f"{bronze_base}/{delivery_csv_name}"
dbutils.fs.cp(landing_delivery_notes_path, delivery_csv_dst)
if delete_from_landing:
    dbutils.fs.rm(landing_delivery_notes_path)
log(f"Copied delivery CSV to: {delivery_csv_dst}")

# COMMAND ----------
# Auto Loader for Excel (.xlsx)

excel_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "binaryFile")
    .option("cloudFiles.schemaLocation", f"{checkpoint_base}/excel/schema")
    .option("cloudFiles.includeExistingFiles", "true")
    .option("pathGlobFilter", "*.xlsx")
    .load(landing_excel_path)
)

excel_query = (
    excel_stream.writeStream
    .foreachBatch(lambda df, batchId: copy_batch(df, bronze_excel_path))
    .option("checkpointLocation", f"{checkpoint_base}/excel/checkpoint")
    .trigger(availableNow=True)
    .start()
)
log(f"Started Excel stream -> {bronze_excel_path}")

# COMMAND ----------
# Auto Loader for Drawings (.DWG)

drawings_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "binaryFile")
    .option("cloudFiles.schemaLocation", f"{checkpoint_base}/drawings/schema")
    .option("cloudFiles.includeExistingFiles", "true")
    .option("pathGlobFilter", "*.[dD][wW][gG]")
    .load(landing_drawings_path)
)

drawings_query = (
    drawings_stream.writeStream
    .foreachBatch(lambda df, batchId: copy_batch(df, bronze_drawings_path))
    .option("checkpointLocation", f"{checkpoint_base}/drawings/checkpoint")
    .trigger(availableNow=True)
    .start()
)
log(f"Started Drawings stream -> {bronze_drawings_path}")

# COMMAND ----------
# Auto Loader for Models (.STEP)

models_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "binaryFile")
    .option("cloudFiles.schemaLocation", f"{checkpoint_base}/models/schema")
    .option("cloudFiles.includeExistingFiles", "true")
    .option("pathGlobFilter", "*.[sS][tT][eE][pP]")
    .load(landing_models_path)
)

models_query = (
    models_stream.writeStream
    .foreachBatch(lambda df, batchId: copy_batch(df, bronze_models_path))
    .option("checkpointLocation", f"{checkpoint_base}/models/checkpoint")
    .trigger(availableNow=True)
    .start()
)
log(f"Started Models stream -> {bronze_models_path}")

# COMMAND ----------
# Espera a que terminen los micro-batches

excel_query.awaitTermination()
drawings_query.awaitTermination()
models_query.awaitTermination()

log_path = f"{bronze_base}/logs"
log_file = f"{log_path}/autoloader_{run_date}_batch_{batch_id}.log"
dbutils.fs.mkdirs(log_path)
dbutils.fs.put(log_file, "\n".join(log_lines) + "\n", overwrite=True)
print(f"Log written to: {log_file}")

# COMMAND ----------
# Escribir log a Delta Table de control

control_path = f"{abfss_base}{bronze_root}/_control/autoloader_logs"
control_table = f"{uc_catalog}.{uc_schema}.autoloader_logs"

rows = []
for line in log_lines:
    ts, msg = line.split(" | ", 1)
    rows.append((ts, msg, project, source, run_date, batch_id, job_id, run_id))

log_df = spark.createDataFrame(
    rows,
    ["ts_utc", "message", "project", "source", "run_date", "batch_id", "job_id", "run_id"]
)

(
    log_df.write
    .format("delta")
    .mode("append")
    .save(control_path)
)

print(f"Control log written to Delta: {control_path}")

# Registrar Delta Table en Unity Catalog si existe el schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {uc_catalog}.{uc_schema}")
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {control_table}
    USING DELTA
    LOCATION '{control_path}'
    """
)
print(f"Control log registered in UC: {control_table}")

# COMMAND ----------
# Task Values para pipeline
try:
    dbutils.jobs.taskValues.set(key="bronze_base", value=bronze_base)
    dbutils.jobs.taskValues.set(key="bronze_excel_path", value=bronze_excel_path)
    dbutils.jobs.taskValues.set(key="batch_id", value=batch_id)
    dbutils.jobs.taskValues.set(key="run_date", value=run_date)
    dbutils.jobs.taskValues.set(key="project", value=project)
    dbutils.jobs.taskValues.set(key="source", value=source)
    print("Task Values set for downstream tasks.")
except Exception as e:
    print(f"Task Values not set (non-job context): {e}")
